# CANDIDATE FEATURE IDENTIFICATION
The purpose of this notebook is to aggregate in one location the potential features to include in the training set. 
Inputs: External Dataset Parquet Files. 
Desired Outputs: 
- Of redundant variables, select the most reliable and informative one. 
- Understand how to calculate any computed features.

Notes
- Use a separate .py module under src/external_dataset_name/ingest.py to pull in the data with API or other method and write to data/interim/external_dataset_name.parquet. 
- This notebook should read in just a small sample of each dataset for feature exploration. Reading in the full dataset may crash the Jupyter kernel. 

For each feature, attempt to identify:
- time of observation
- unit of observation / what it actually is
- geographic scope (i.e. is it at the census block group, tract, or zip code level?)
- missingness
- what preprocessing is required (e.g. geocoding to census block group, converting a raw count to density)

In [2]:
from pathlib import Path

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

In [3]:
pd.set_option('display.max_columns', None)

## Lottery and Sales Data (for reference when merging)

In [4]:
lottery_locations_file_path = Path(".").resolve().parent / "data/final/location_data.parquet"

#locations_pq = pq.ParquetFile(lottery_locations_file_path)
#locations_first_100_rows = next(locations_pq.iter_batches(batch_size=100))
#locations_df = pa.Table.from_batches([locations_first_100_rows]).to_pandas()

locations_df = pd.read_parquet(str(lottery_locations_file_path))

In [5]:
locations_df.sample(5)

,agtno,busnm,busaddr,buscity,buszip,chain,bustype,longitude,latitude,geometry,bg_geoid,tract_geoid
7634,77370,OSSINING FOOD & GAS,2 PLEASANTVILLE RD,OSSINING,10562,0,CONV. STORE,-73.84932,41.16674,b'\x01\x01\x00\x00\x00\xae\xf5EB[vR\xc0\xafw\x...,361190134022,36119013402
15278,115215,GEROWS DAIRY STORE,98 OLD ITHACA RD,HORSEHEADS,14845,0,CONV. STORE,-76.81733,42.17397,"b'\x01\x01\x00\x00\x00\x8a\x02}""O4S\xc0\x18>""\...",360150107023,36015010702
2287,105159,JAMAICA GOURMET DELI 1 INC,16834 JAMAICA AVE,JAMAICA,11432,0,GROCERY STORE,-73.79135,40.70674,"b'\x01\x01\x00\x00\x00""lxz\xa5rR\xc05c\xd1tvZD@'",360810460003,36081046000
9456,99526,HAAGEN DAZS,451 WEST 39TH ST,NEW YORK,10018,0,MISCELLANEOUS,-73.99547,40.75715,b'\x01\x01\x00\x00\x00\x8b\x89\xcd\xc7\xb5\x7f...,360610115002,36061011500
1846,101186,KHABRA ENTERPRISES INC,837 NASH RD,NORTH TONAWANDA,14120,0,CONV. STORE,-78.86541,43.04566,b'\x01\x01\x00\x00\x00h\xe8\x9f\xe0b\xb7S\xc0 ...,360630246003,36063024600


In [6]:
lottery_sales_file_path = Path(".").resolve().parent / "data/final/sales_data.parquet"

#sales_pq = pq.ParquetFile(lottery_sales_file_path)
#sales_first_100_rows = next(sales_pq.iter_batches(batch_size=1000))
#sales_df = pa.Table.from_batches([sales_first_100_rows]).to_pandas()

sales_df = pd.read_parquet(str(lottery_sales_file_path))

In [7]:
sales_df.sample(5)

,agtno,total_sales,first_date,last_date,first_pos_date,last_pos_date,pos_sales_days,num_cal_days,num_active_days,days_before_first_sale,days_after_last_sale,avg_cal_daily_sales,avg_active_daily_sales,use_active_period,avg_daily_sales
5717,114218,627368.5,2025-01-01,2025-12-31,2025-01-01,2025-12-31,365.0,365,365.0,0.0,0.0,1718.817808,1718.817808,False,1718.817808
8658,37925,25104.5,2025-01-02,2025-12-31,2025-01-02,2025-12-31,331.0,364,364.0,0.0,0.0,68.968407,68.968407,False,68.968407
12618,79531,2848756.0,2025-01-01,2025-12-31,2025-01-01,2025-12-31,365.0,365,365.0,0.0,0.0,7804.810959,7804.810959,False,7804.810959
12399,78189,1490862.5,2025-01-01,2025-12-31,2025-01-01,2025-12-31,365.0,365,365.0,0.0,0.0,4084.554795,4084.554795,False,4084.554795
750,102892,101874.0,2025-01-01,2025-09-22,2025-01-01,2025-09-22,238.0,265,265.0,0.0,0.0,384.430189,384.430189,False,384.430189


## Overture Maps Data

In [8]:
overture_file_path = Path(".").resolve().parent / "data/interim/external/merged_with_loc/location_data_overture.parquet"

overture_pq = pq.ParquetFile(overture_file_path)
overture_first_100_rows = next(overture_pq.iter_batches(batch_size=100))
overture_df = pa.Table.from_batches([overture_first_100_rows]).to_pandas()

In [9]:
overture_df.sample(5)

,agtno,busnm,busaddr,buscity,buszip,longitude,latitude,geometry,bg_geoid,tract_geoid,gas_station_within_0.5_mi,gas_station_within_1_mi,arts_and_entertainment_within_0.5_mi,arts_and_entertainment_within_1_mi,shopping_within_0.5_mi,shopping_within_1_mi,social_or_community_service_within_0.5_mi,social_or_community_service_within_1_mi,restaurant_within_0.5_mi,restaurant_within_1_mi,bar_within_0.5_mi,bar_within_1_mi
29,76146,RITE AID #10642,695 EAST JERICHO TURNPIKE,HUNTINGTON STATION,11746,-73.38412,40.83514,b'\x01\x01\x00\x00\x00To\rl\x95XR\xc0m\xca\x15...,361031114021,36103111402,0,1,0,0,0,0,0,1,0,0,0,1
87,100319,MIA TRAVEL MULTISERVICE,5 E CLARK PLACE,BRONX,10452,-73.91877,40.83828,b'\x01\x01\x00\x00\x00\xf0\xa2\xaf \xcdzR\xc0\...,360050221013,36005022101,2,9,2,10,0,9,17,47,9,21,3,7
38,65723,A-PLUS #08664,885 JOHNSON AVE,RONKONKOMA,11779,-73.12397,40.80767,b'\x01\x01\x00\x00\x00\xd9\xeb\xdd\x1f\xefGR\x...,361031458082,36103145808,2,7,0,1,0,4,0,5,0,0,0,1
99,113075,RAINBOW DELI GROCERY CORP,38-15 GREENPOINT AVE,LONG ISLAND CITY,11101,-73.92779,40.73838,b'\x01\x01\x00\x00\x00\x93\xe3N\xe9`{R\xc0\x9f...,360810189003,36081018900,7,11,0,5,3,8,5,23,4,17,3,10
72,104406,S & H FOOD MART,361 WALNUT ST,ELMIRA,14901,-76.81823,42.08863,b'\x01\x01\x00\x00\x00\xca\xa6\\\xe1]4S\xc0\xe...,360150005002,36015000500,2,4,1,1,1,5,3,19,1,2,0,2


## LEHD Data

In [10]:
rac_file_path = Path(".").resolve().parent / "data/interim/external/merged_with_loc/location_data_rac.parquet"

rac_pq = pq.ParquetFile(rac_file_path)
rac_first_100_rows = next(rac_pq.iter_batches(batch_size=100))
rac_df = pa.Table.from_batches([rac_first_100_rows]).to_pandas()

In [11]:
rac_df.sample(2)

,agtno,busnm,busaddr,buscity,buszip,longitude,latitude,geometry,bg_geoid,tract_geoid,rac_c000,rac_ce01,rac_ce02,rac_ce03
6,77603,SUPER D,357 N MAIN ST,FREEPORT,11520,-73.58764,40.66898,b'\x01\x01\x00\x00\x00\x92t\xcd\xe4\x9beR\xc0\...,360594141005,36059414100,480.0,84.0,159.0,237.0
21,11530,PAY-O-MATIC CORP #103,160 ROCKAWAY AVE,BROOKLYN,11233,-73.91079,40.67780,b'\x01\x01\x00\x00\x00\x87\xe1#bJzR\xc0I\x9d\x...,360470369002,36047036900,572.0,106.0,185.0,281.0


In [12]:
wac_file_path = Path(".").resolve().parent / "data/interim/external/merged_with_loc/location_data_wac.parquet"

wac_pq = pq.ParquetFile(wac_file_path)
wac_first_100_rows = next(wac_pq.iter_batches(batch_size=100))
wac_df = pa.Table.from_batches([wac_first_100_rows]).to_pandas()

In [13]:
wac_df.sample(2)

,agtno,busnm,busaddr,buscity,buszip,longitude,latitude,geometry,bg_geoid,tract_geoid,wac_c000,wac_cns07,wac_cns18,wac_ce01,wac_ce02,wac_ce03
41,78564,MOUNT MART INC,140 W SANDFORD BLVD,MOUNT VERNON,10550,-73.83829,40.89820,b'\x01\x01\x00\x00\x00\x16\xa4\x19\x8b\xa6uR\x...,361190028002,36119002800,67.0,0.0,6.0,16.0,18.0,33.0
83,108868,WEST44TH STREET NEWSSTAND LLC,300 WEST 44TH STREET,NEW YORK,10036,-73.98944,40.75879,b'\x01\x01\x00\x00\x00\xacV&\xfcR\x7fR\xc0\x13...,360610121012,36061012101,6172.0,13.0,1567.0,876.0,1216.0,4080.0


## AADT Data

In [14]:
aadt_file_path = Path(".").resolve().parent / "data/interim/external/merged_with_loc/data_aadt_2025.parquet"

aadt_pq = pq.ParquetFile(aadt_file_path)
aadt_first_100_rows = next(aadt_pq.iter_batches(batch_size=100))
aadt_df = pa.Table.from_batches([aadt_first_100_rows]).to_pandas()

In [15]:
aadt_df.sample(5)

,agtno,busnm,busaddr,buscity,buszip,longitude,latitude,bg_geoid,tract_geoid,index_right,aadt_count,aadt_road_name,aadt_functional_class,aadt_lanes,dist_to_read_m
53,115699,NY GRILL & DELI INC,1938 MADISON AVE,NEW YORK,10035,-73.94096,40.80533,360610198001,36061019800,54008.0,57017.0,WILLIS AVE BRG,14,4:0||||,913.844131
17,40927,KWIK FILL #025,8 W STATE ST,BINGHAMTON,13901,-75.90666,42.10857,360070005003,36007000500,55222.0,60857.0,NaN,12,3:2||||,543.834664
55,115666,THE BOTTLE ZONE,1875 LEXINGTON AVE,NEW YORK,10035,-73.94122,40.79918,360610182001,36061018200,54008.0,57017.0,WILLIS AVE BRG,14,4:0||||,891.025336
4,51646,KWIK FILL #335,380 N WINTON RD,ROCHESTER,14610,-77.55099,43.15274,360550076001,36055007600,55612.0,77615.0,NaN,12,2:2|3:2|3:3||,962.929010
31,14345,EL BARRIO GROCERY,1858 LEXINGTON AVE,NEW YORK,10029,-73.94205,40.79804,360610182001,36061018200,54008.0,57017.0,WILLIS AVE BRG,14,4:0||||,1002.592798


## EPA Data

In [16]:
epa_file_path = Path(".").resolve().parent / "data/interim/external/merged_with_loc/location_data_epa.parquet"

epa_pq = pq.ParquetFile(epa_file_path)
epa_first_100_rows = next(epa_pq.iter_batches(batch_size=100))
epa_df = pa.Table.from_batches([epa_first_100_rows]).to_pandas()

In [17]:
epa_df.sample(5)

,agtno,busnm,busaddr,buscity,buszip,chain,bustype,longitude,latitude,geometry,bg_geoid,tract_geoid,OBJECTID,STATEFP,COUNTYFP,TRACTCE,BLKGRPCE,D1A,D1B,D1C,D1C8_RET,D1C8_OFF,D1C8_IND,D1C8_SVC,D1C8_ENT,D1C8_ED,D1C8_HLTH,D1C8_PUB,D3A,NatWalkInd,STATEFP2,COUNTYFP2,TRACTCE2,BLKGRPCE2
46,91274,KINGS NAILS EIGHT LLC,3622 WHITE PLAINS RD,BRONX,10467,0,SPECIALTY(NON FOOD),-73.864800,40.879860,"b'\x01\x01\x00\x00\x00,e\x19\xe2XwR\xc0\x7f\x8...",360050380003,36005038000,160802,36,5,38000,3,53.551564,162.224876,9.173185,1.074337,6.115456,0.743772,0.247924,0.743772,0.000000,0.247924,0.000000,32.274792,14.500000,36,005,038000,3
98,91896,ASTORIA GOOD CONV STORE INC,30-16 30TH AVE,ASTORIA,11102,0,STATIONERY-CARD,-73.921640,40.767010,b'\x01\x01\x00\x00\x00\xde\xabV&\xfczR\xc0<\xd...,360810063001,36081006300,148955,36,81,6300,1,48.915650,84.162563,40.315096,16.740364,3.993114,2.457301,2.610882,1.459023,3.762742,9.291670,0.000000,29.010466,17.833333,36,081,006300,1
35,78975,BULLVILLE FOOD MART,2887 ROUTE 17K,BULLVILLE,10915,0,CONV. STORE,-74.358904,41.541798,"b'\x01\x01\x00\x00\x00\x1d\xcb""G\xf8\x96R\xc0w...",360710109023,36071010902,153458,36,71,10902,3,0.134975,0.404333,0.059496,0.008880,0.000296,0.026048,0.016872,0.000000,0.000000,0.007400,0.000000,2.878447,5.166667,36,071,010902,3
30,88265,CIBAO GROCERY CORP,89-04 86TH ST,WOODHAVEN,11421,0,DELI-GROCERY,-73.857000,40.689390,b'\x01\x01\x00\x00\x00\x02+\x87\x16\xd9vR\xc0@...,360810008001,36081000800,149529,36,81,800,1,21.613733,64.341723,5.358026,0.136221,0.000000,0.544884,0.136221,0.000000,0.000000,0.227035,4.313665,31.268510,13.833333,36,081,000800,1
53,100319,MIA TRAVEL MULTISERVICE,5 E CLARK PLACE,BRONX,10452,0,TRAVEL AGENT,-73.918770,40.838280,b'\x01\x01\x00\x00\x00\xf0\xa2\xaf \xcdzR\xc0\...,360050221013,36005022101,153733,36,5,22101,3,98.072665,312.244844,11.246107,2.976911,1.157687,2.315375,2.976911,1.819223,0.000000,0.000000,0.000000,27.642534,14.166667,36,005,022101,3


## ACS Data

In [18]:
acs_file_path = Path(".").resolve().parent / "data/interim/external/merged_with_loc/location_data_acs.parquet"

acs_pq = pq.ParquetFile(acs_file_path)
acs_first_100_rows = next(acs_pq.iter_batches(batch_size=100))
acs_df = pa.Table.from_batches([acs_first_100_rows]).to_pandas()

In [19]:
acs_df.sample(5)

,agtno,busnm,busaddr,buscity,buszip,chain,bustype,longitude,latitude,geometry,bg_geoid,tract_geoid,median_household_income,households_total,total_population,age_18_29_proportion,age_30_64_proportion,age_65_plus_proportion,poverty_rate,bachelors_degree_proportion_25plus,post_grad_degree_proportion_25plus,in_labor_force_proportion_16plus,unemployed_proportion_16plus,owner_occupied_proportion,renter_occupied_proportion,family_households_proportion,nonfamily_multiperson_households_proportion,living_alone_households_proportion
61,94655,WILLY'S DELI CORP,1356 ROCKAWAY PKWY,BROOKLYN,11236,0,DELI-GROCERY,-73.90388,40.64652,b'\x01\x01\x00\x00\x00\x8b\xe0\x7f+\xd9yR\xc0\...,360470968001,36047096800,105292.0,195.0,486.0,0.102881,0.594650,0.302469,0.053398,0.180258,0.199571,0.475149,0.023857,0.687179,0.312821,0.841026,0.061538,0.097436
68,76475,QUICKLEES #12,349 W MORRIS ST,BATH,14810,290,CONV. STORE,-77.33299,42.33839,b'\x01\x01\x00\x00\x00H\xf9I\xb5OUS\xc0\x8f\xa...,361019611001,36101961100,58650.0,623.0,1043.0,0.129434,0.687440,0.183126,0.116599,0.165165,0.070070,0.701869,0.000000,0.815409,0.184591,0.555377,0.097913,0.346709
65,110085,US FOOD MART DELI & GROCERY IN,952 CRANE ST,SCHENECTADY,12303,0,DELI-GROCERY,-73.94798,42.79824,b'\x01\x01\x00\x00\x00\xc6PN\xb4\xab|R\xc0\xf5...,360930214002,36093021400,79479.0,207.0,532.0,0.259398,0.592105,0.148496,0.180624,0.000000,0.105590,0.744361,0.189850,0.574879,0.425121,0.594203,0.342995,0.062802
23,106447,STUDENT FINEST CONVENIENT,22 MAIN ST,CORTLAND,13045,0,CONV. STORE,-76.18044,42.60048,b'\x01\x01\x00\x00\x00\xfa\xb86T\x8c\x0bS\xc0w...,360239706002,36023970600,43558.0,535.0,823.0,0.394897,0.471446,0.133657,0.340233,0.047218,0.150084,0.571081,0.000000,0.031776,0.968224,0.231776,0.155140,0.613084
9,47609,THE BLUE HERON,12050 NYS RT 12 E,CHAUMONT,13622,0,RESTAURANT-HOTEL,-76.13022,44.06718,b'\x01\x01\x00\x00\x00?RD\x86U\x08S\xc0\xffx\x...,360450603023,36045060302,79286.0,376.0,742.0,0.084906,0.571429,0.343666,0.144989,0.191549,0.150704,0.530090,0.015365,0.957447,0.042553,0.723404,0.029255,0.247340


## Master Data Sample

In [20]:
master_file_path = Path(".").resolve().parent / "data/final/master_data.parquet"
master_df = pd.read_parquet(str(master_file_path))

master_df.sample(5)

,agtno,busnm,busaddr,buscity,buszip,chain,bustype,longitude,latitude,geometry,bg_geoid,tract_geoid,aadt_count,aadt_road_name,aadt_functional_class,aadt_lanes,dist_to_read_m,median_household_income,households_total,total_population,age_18_29_proportion,age_30_64_proportion,age_65_plus_proportion,poverty_rate,bachelors_degree_proportion_25plus,post_grad_degree_proportion_25plus,in_labor_force_proportion_16plus,unemployed_proportion_16plus,owner_occupied_proportion,renter_occupied_proportion,family_households_proportion,nonfamily_multiperson_households_proportion,living_alone_households_proportion,D1A,D1B,D1C,D1C8_RET,D1C8_OFF,D1C8_IND,D1C8_SVC,D1C8_ENT,D1C8_ED,D1C8_HLTH,D1C8_PUB,D3A,NatWalkInd,gas_station_within_0.5_mi,gas_station_within_1_mi,arts_and_entertainment_within_0.5_mi,arts_and_entertainment_within_1_mi,shopping_within_0.5_mi,shopping_within_1_mi,social_or_community_service_within_0.5_mi,social_or_community_service_within_1_mi,restaurant_within_0.5_mi,restaurant_within_1_mi,bar_within_0.5_mi,bar_within_1_mi,rac_c000,rac_ce01,rac_ce02,rac_ce03,wac_c000,wac_cns07,wac_cns18,wac_ce01,wac_ce02,wac_ce03
4460,37436,RAVENA SHOP N SAVE,2481 ROUTE 9W,RAVENA,12143,0,GROCERY STORE,-73.81638,42.46696,b'\x01\x01\x00\x00\x00\xf1F\xe6\x91?tR\xc0$EdX...,360010144012,36001014401,NaN,NaN,NaN,NaN,NaN,111016.0,1044.0,2178.0,0.213499,0.657484,0.129017,0.033438,0.181768,0.168983,0.681897,0.021323,0.876437,0.123563,0.720307,0.020115,0.259579,0.061993,0.120835,0.029038,0.009793,0.001703,0.012688,0.000766,0.003236,0.000000,0.000852,0.000000,2.541229,5.666667,0.0,0.0,0.0,0.0,0.0,0.0,2.0,2.0,0.0,0.0,0.0,0.0,754.0,110.0,165.0,479.0,439.0,151.0,0.0,49.0,83.0,307.0
5212,47591,BJ'S WHOLESALE CLUB #168,400 JAY SCUTTI BOULEVARD,ROCHESTER,14623,67,CLUB STORE,-77.63227,43.09130,b'\x01\x01\x00\x00\x00|\x0f\x97\x1cwhS\xc0\xf9...,360550131041,36055013104,19874.0,NaN,14,2:2||||,478.511196,55363.0,390.0,524.0,0.318702,0.541985,0.139313,0.306502,0.148073,0.269777,0.780669,0.061338,0.248718,0.751282,0.407692,0.033333,0.558974,0.955703,2.179159,5.599363,3.049845,0.088925,0.736810,0.320522,1.242023,0.145603,0.009772,0.005863,13.617816,12.500000,1.0,3.0,0.0,0.0,2.0,7.0,1.0,2.0,2.0,3.0,0.0,0.0,461.0,133.0,116.0,212.0,8770.0,3022.0,1565.0,2369.0,2765.0,3636.0
2028,108635,PEACE INTERNATIONAL MARKET,3240 RICHMOND TERRACE,STATEN ISLAND,10303,0,DELI-GROCERY,-74.16613,40.64032,b'\x01\x01\x00\x00\x00\x9b8\xb9\xdf\xa1\x8aR\x...,360850319022,36085031902,2730.0,Harbor Rd,19,1:1||||,562.045032,142895.0,666.0,1642.0,0.213764,0.662607,0.123630,0.118750,0.130790,0.168937,0.675423,0.007589,0.797297,0.202703,0.827327,0.016517,0.156156,8.827414,25.169816,11.182590,0.197763,0.000000,10.787064,0.017978,0.000000,0.000000,0.179784,0.000000,20.873868,14.333333,0.0,1.0,0.0,0.0,0.0,0.0,0.0,3.0,0.0,1.0,0.0,0.0,666.0,114.0,195.0,357.0,418.0,3.0,3.0,12.0,32.0,374.0
13603,112484,ROCKLAND MOBIL MART INC,1904 ROUTE 44,PLEASANT VALLEY,12569,0,CONV. STORE,-73.79783,41.75255,"b'\x01\x01\x00\x00\x00""q\x8f\xa5\x0fsR\xc0jM\x...",360271300051,36027130005,1905.0,TRAVER RD,19,1:1||||,1317.367350,90217.0,638.0,1299.0,0.181678,0.665897,0.152425,0.034381,0.186690,0.133103,0.704678,0.008772,0.846395,0.153605,0.630094,0.047022,0.322884,0.147706,0.329885,0.026421,0.007046,0.004026,0.006794,0.006291,0.001258,0.000000,0.001007,0.000000,4.319728,5.500000,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,752.0,132.0,144.0,476.0,112.0,33.0,2.0,29.0,35.0,48.0
879,73779,RITE AID #1019,133 ROUTE 303,VALLEY COTTAGE,10989,296,DRUG STORE,-73.94245,41.11922,b'\x01\x01\x00\x00\x00_\x07\xce\x19Q|R\xc0\xba...,360870111011,36087011101,NaN,NaN,NaN,NaN,NaN,136925.0,521.0,1210.0,0.174380,0.417355,0.408264,0.000000,0.374081,0.306066,0.673753,0.000000,0.834933,0.165067,0.660269,0.044146,0.295585,1.259142,2.799039,4.792680,0.224037,0.354488,2.285740,1.052121,0.175826,0.104929,0.587032,0.008508,11.872231,9.500000,1.0,2.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0,1.0,538.0,81.0,101.0,356.0,1220.0,91.0,14.0,155.0,255.0,810.0
